# EDA & Evaluation - Basketball Action Recognition

Notebook này thực hiện:
1. Khám phá dữ liệu (EDA) trên `data/data.csv`
2. Train lại model SVM (giống `src/training/train_svm.py`) để có object model/y_pred trong notebook
3. Vẽ confusion matrix, classification report dạng biểu đồ
4. So sánh với phiên bản v2 (position + velocity)

## 1. Import & Load dữ liệu

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

BASE_DIR = os.path.dirname(os.getcwd())  # notebooks/ -> Project-I/
DATA_PATH = os.path.join(BASE_DIR, "data", "data.csv")
DATA_V2_PATH = os.path.join(BASE_DIR, "data", "data_v2.csv")

data = pd.read_csv(DATA_PATH)
data.head()

## 2. Khám phá dữ liệu (EDA)

In [ ]:
print("Tổng số sample:", len(data))
print("Số features:", data.shape[1] - 1)
print("\nPhân phối các class:")
print(data["label"].value_counts())

In [ ]:
# Biểu đồ phân phối số lượng sample theo class
plt.figure(figsize=(6, 4))
sns.countplot(data=data, x="label", order=data["label"].value_counts().index)
plt.title("Số lượng sample theo class")
plt.xlabel("Action")
plt.ylabel("Số sample")
plt.show()

In [ ]:
# Kiểm tra giá trị thiếu / phạm vi giá trị của vài cột đầu
print(data.isnull().sum().sum(), "giá trị bị thiếu")
data.describe().iloc[:, :8]

## 3. Train model (giống `train_svm.py`)

In [ ]:
X = data.drop("label", axis=1)
y = data["label"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

model = SVC(kernel="rbf", C=10, gamma="scale", probability=True)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

## 4. Confusion Matrix (trực quan)

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=model.classes_)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_)
disp.plot(ax=ax, cmap="Blues", colorbar=False)
plt.title("Confusion Matrix - SVM v1 (position only)")
plt.show()

## 5. So sánh với model khác (Random Forest)

So sánh nhanh với một thuật toán khác trên cùng dữ liệu để xem SVM có phải lựa chọn tốt.

In [ ]:
rf_model = RandomForestClassifier(n_estimators=200, random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, rf_pred))
print(classification_report(y_test, rf_pred))

In [ ]:
# Bảng so sánh tổng hợp
results = pd.DataFrame({
    "Model": ["SVM (v1, position only)", "Random Forest (position only)"],
    "Accuracy": [accuracy_score(y_test, y_pred), accuracy_score(y_test, rf_pred)],
})
results

## 6. So sánh v1 vs v2 (position + velocity)

Nếu đã chạy `extract_keypoints_v2.py` và có `data/data_v2.csv`, train lại
SVM trên data_v2 để so sánh.

In [ ]:
if os.path.exists(DATA_V2_PATH):
    data_v2 = pd.read_csv(DATA_V2_PATH)

    X2 = data_v2.drop("label", axis=1)
    y2 = data_v2["label"]

    scaler2 = StandardScaler()
    X2_scaled = scaler2.fit_transform(X2)

    X2_train, X2_test, y2_train, y2_test = train_test_split(
        X2_scaled, y2, test_size=0.2, random_state=42, stratify=y2
    )

    model_v2 = SVC(kernel="rbf", C=10, gamma="scale", probability=True)
    model_v2.fit(X2_train, y2_train)
    y2_pred = model_v2.predict(X2_test)

    acc_v2 = accuracy_score(y2_test, y2_pred)
    print("SVM v2 (position + velocity) Accuracy:", acc_v2)

    comparison = pd.DataFrame({
        "Version": ["v1 (position only, 132 features)", "v2 (position + velocity, 264 features)"],
        "Accuracy": [accuracy_score(y_test, y_pred), acc_v2],
    })

    display(comparison)

    plt.figure(figsize=(5, 4))
    sns.barplot(data=comparison, x="Version", y="Accuracy")
    plt.ylim(0.8, 1.0)
    plt.title("So sánh Accuracy: v1 vs v2")
    plt.xticks(rotation=15)
    plt.show()
else:
    print("Chưa có data_v2.csv - chạy extract_keypoints_v2.py trước.")

## 7. Kết luận

- Model SVM v1 (position only) đạt accuracy ~0.986, hiệu suất tốt cho 4
  class (dribbling, shooting, defense, idle).
- Thử nghiệm v2 (thêm velocity) cho accuracy thấp hơn (~0.944), do cách
  lấy mẫu dataset không liên tục theo thời gian khiến velocity trở
  thành nhiễu.
- Random Forest cho kết quả tương đương/thấp hơn SVM trên cùng dữ liệu.
- Hướng cải tiến tiếp theo: trích keypoints theo frame liên tiếp thực sự
  (fps cố định) để velocity/sequence model có ý nghĩa hơn.